# 🤖 Ollama + Qwen 3.8 27B TURBO Uncensored sur Kaggle Notebooks

Ce notebook configure automatiquement un environnement LLM haute performance avec:
- **Ollama** - Runtime d'inférence locale ultra-rapide
- **Modèle 27B**: `hf.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF:Q4_K_M` (~17 Go, Q4_K_M)
- **Nettoyage & Gestion du disque** - Option de suppression d'anciens modèles (`ollama rm`) pour libérer l'espace Kaggle
- **Tunnel Cloudflare** - Accès public distant gratuit (URL trycloudflare.com)

⚡ **Recommandation**: Activez l'accélérateur **GPU T4 x2** dans le panneau Settings de droite sur Kaggle.

🚀 **Ou lancez tout en 1 seule ligne de commande dans Kaggle** :
```bash
!curl -fsSL https://raw.githubusercontent.com/yomix90/free-kaggle-llm/main/setup_kaggle.sh | bash
```
---


## 📋 Étape 1: Installation des dépendances système

Installation de Zstandard et mise à jour des paquets.

In [ ]:
import subprocess
import time

print("[1/6] Mise à jour du système...")
subprocess.run("apt-get update -qq", shell=True, capture_output=True)
print("✓ Système mis à jour")

print("\n[2/6] Installation de Zstandard...")
subprocess.run("apt-get install -y -qq zstd", shell=True, capture_output=True)
print("✓ Zstandard installé")


## 🔧 Étape 2: Installation d'Ollama

In [ ]:
print("[3/6] Installation d'Ollama...")
result = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("✓ Ollama installé avec succès")
else:
    print(f"⚠️ {result.stderr}")

res = subprocess.run("ollama --version", shell=True, capture_output=True, text=True)
print(f"Version: {res.stdout.strip()}")


## 🚀 Étape 3: Démarrage d'Ollama en arrière-plan

In [ ]:
print("[4/6] Démarrage du serveur Ollama...")
subprocess.run("pkill -f 'ollama serve'", shell=True, capture_output=True)
time.sleep(1)
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print(f"✓ Ollama actif (PID: {ollama_process.pid})")


## 🧹 Étape 4: Gestion de l'espace disque et suppression d'anciens modèles

> **Pourquoi supprimer un modèle ?**
> Le modèle **Qwen 3.8 27B** pèse environ **17 Go**. Le disque d'un notebook Kaggle étant limité (~20 Go à 50 Go), avoir un autre gros modèle déjà téléchargé peut saturer l'espace disque (`no space left on device`).
> Exécutez la cellule ci-dessous pour vérifier l'espace disponible et supprimer un ancien modèle si nécessaire.

In [ ]:
# 1. Vérifier l'espace disque disponible
!df -h /

# 2. Lister les modèles actuels et leur taille
!ollama list

# 3. Supprimer un ancien modèle si vous manquez d'espace (décommentez selon votre besoin) :
# !ollama rm hf.co/theLittleStone/Qwen3.6-27B-AEON-Ultimate-Uncensored-MTP-i1-GGUF:Q4_K_M
# !ollama rm qwen:7b


## 🧠 Étape 5: Téléchargement du modèle 27B

Téléchargement de `hf.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF:Q4_K_M` (~17 Go).

In [ ]:
MODEL = "hf.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF:Q4_K_M"
print(f"Téléchargement du modèle {MODEL}...")
print("(Prévoyez environ 5 à 15 minutes selon la bande passante Kaggle)\n")
!ollama pull {MODEL}
print("\n✓ Téléchargement terminé !")
!df -h /


## ✅ Étape 6: Test du modèle

In [ ]:
prompt = "Bonjour ! Présente-toi et donne tes capacités principales."
!echo "$prompt" | ollama run hf.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF:Q4_K_M


## 📚 Utiliser l'API REST locale

In [ ]:
import requests

url = "http://localhost:11434/api/generate"
data = {
    "model": "hf.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF:Q4_K_M",
    "prompt": "Explique brièvement ce qu'est le fine-tuning Heretic Uncensored.",
    "stream": False
}

resp = requests.post(url, json=data)
print(resp.json().get("response"))


## 🌐 Optionnel: Tunnel Cloudflare pour accès distant public

In [ ]:
# Installation de Cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
print("✓ Cloudflared prêt")

# Démarrage du tunnel
cloudflared = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(8)
for _ in range(40):
    line = cloudflared.stdout.readline()
    if line:
        print(line.rstrip())
        if "trycloudflare.com" in line:
            print("\n✓ URL PUBLIQUE ACTIVE CI-DESSUS !")


## 🔌 Classe Helper Python réutilisable pour Ollama

In [ ]:
import requests

class OllamaClient:
    def __init__(self, model="hf.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF:Q4_K_M", base_url="http://localhost:11434"):
        self.model = model
        self.base_url = base_url

    def generate(self, prompt, temperature=0.7):
        url = f"{self.base_url}/api/generate"
        data = {"model": self.model, "prompt": prompt, "temperature": temperature, "stream": False}
        r = requests.post(url, json=data, timeout=300)
        return r.json()["response"]

    def delete_model(self, model_name):
        """Supprime un modèle via l'API pour libérer de l'espace"""
        url = f"{self.base_url}/api/delete"
        r = requests.delete(url, json={"model": model_name})
        return r.status_code == 200

    def list_models(self):
        url = f"{self.base_url}/api/tags"
        return requests.get(url).json().get("models", [])

client = OllamaClient()
print(client.generate("Donne 3 conseils de programmation Python."))
